In [3]:
import geopandas as gpd
import numpy as np
import pandas as pd
import os
import json
from pathlib import Path
import datetime
from shapely import wkt
from shapely.geometry import LineString
from tqdm import tqdm

from joblib import Parallel, delayed
import multiprocessing

import trackintel as ti
from trackintel.analysis.tracking_quality import temporal_tracking_quality

In [4]:
ti.__version__

'1.4.2'

In [5]:
# read file storage
Dataset_file = os.path.join("paths.json")
with open(Dataset_file) as json_file:
    CONFIG = json.load(json_file)

FileNotFoundError: [Errno 2] No such file or directory: 'paths.json'

# Read staypoints

In [ ]:
sp = pd.read_csv(os.path.join(CONFIG[f"raw_mobis"], "sps.csv"))

In [ ]:
# geometry
sp["geometry"] = gpd.GeoSeries.from_wkt(sp["geometry"])
sp = gpd.GeoDataFrame(sp, crs="EPSG:4326", geometry="geometry")

In [ ]:
sp["started_at"] = pd.to_datetime(sp["started_at"], format='mixed', yearfirst=True, utc=True)
sp["finished_at"] = pd.to_datetime(sp["finished_at"], format='mixed', yearfirst=True, utc=True)

In [ ]:
# to trackintel
sp = ti.io.read_staypoints_gpd(sp)

In [ ]:
type(sp)

In [ ]:
user_max_tracking = sp.groupby("user_id")["finished_at"].max()
user_max_tracking[user_max_tracking.dt.tz_localize(None) >  np.datetime64('2022', "Y")]

# Read triplegs

In [ ]:
# only a subset for testing
# tpls = pd.read_csv(os.path.join(CONFIG["raw_mobis"], "legs.csv"), nrows=100000)

# full, with mode 4
tpls = pd.read_csv(os.path.join(CONFIG["raw_mobis"], "legs.csv"), usecols=[0, 1, 3, 4, 6])
tpls["mode"] = tpls["mode"].apply(lambda x: x[6:])

In [ ]:
# geometry
tpls["geometry"] = gpd.GeoSeries.from_wkt(tpls["geometry"])
tpls = gpd.GeoDataFrame(tpls, crs="EPSG:4326", geometry="geometry")

In [ ]:
# construct linestring from multilinestring
def get_simple_line(multi):
    # multi = wkt.loads(str)
    multicoords = [list(line.coords) for line in multi.geoms]
    simple = LineString([item for sublist in multicoords for item in sublist])
    return simple

MultiLSFlag = tpls.geometry.type == "MultiLineString"
tpls.loc[MultiLSFlag, "geometry"] = tpls.loc[MultiLSFlag, "geometry"].apply(get_simple_line)

In [ ]:
tpls["started_at"] = pd.to_datetime(tpls["started_at"], format='mixed', yearfirst=True, utc=True)
tpls["finished_at"] = pd.to_datetime(tpls["finished_at"], format='mixed', yearfirst=True, utc=True)

In [ ]:
# to trackintel, filter invalid geometry
tpls = ti.io.read_triplegs_gpd(tpls[tpls.geometry.is_valid])

In [ ]:
type(tpls)

## Count trackpoints

In [ ]:
track_count = 0

for i in tqdm(range(len(tpls))):
    track_count+=len(tpls.geometry.iloc[i].coords)

track_count

In [ ]:
(len(sp) * 10 + track_count) / (1000 * 1000 * 1000)

# Final cleaning

In [ ]:
tpls["user_id"].unique().shape, sp["user_id"].unique().shape

In [ ]:
# negative duration records have already been dropped
sp["duration"] = (sp["finished_at"] - sp["started_at"]).dt.total_seconds()
tpls["duration"] = (tpls["finished_at"] - tpls["started_at"]).dt.total_seconds()

In [ ]:
sp = sp.sort_values(by="started_at").reset_index(drop=True)
tpls = tpls.sort_values(by="started_at").reset_index(drop=True)

sp.index.name = "id"
tpls.index.name = "id"

In [ ]:
len(sp), len(tpls)

In [ ]:
tpls.head()

In [ ]:
sp.head()

# Filter duplicates

In [ ]:
def _alter_diff(df):
    df.sort_values(by="started_at", inplace=True)
    df["diff"] = pd.NA
    # for correct dtype
    df["st_next"] = df["started_at"]

    diff = df["started_at"].iloc[1:].reset_index(drop=True) - df["finished_at"].iloc[:-1].reset_index(drop=True)
    df["diff"].iloc[:-1] = diff.dt.total_seconds()
    df["st_next"].iloc[:-1] = df["started_at"].iloc[1:].reset_index(drop=True)

    df.loc[df["diff"] < 0, "finished_at"] = df.loc[df["diff"] < 0, "st_next"]

    df["started_at"], df["finished_at"] = pd.to_datetime(df["started_at"]), pd.to_datetime(df["finished_at"])
    df["duration"] = (df["finished_at"] - df["started_at"]).dt.total_seconds()

    # print(df.loc[df["diff"] < 0])
    df.drop(columns=["diff", "st_next"], inplace=True)
    df.drop(index=df[df["duration"] <= 0].index, inplace=True)

    return df

def filter_duplicates(sp, tpls):

    # merge trips and staypoints
    sp["type"] = "sp"
    tpls["type"] = "tpl"
    df_all = pd.merge(sp, tpls, how="outer")

    df_all = df_all.groupby("user_id", as_index=False).apply(_alter_diff)
    sp = df_all.loc[df_all["type"] == "sp"].drop(columns=["type"])
    tpls = df_all.loc[df_all["type"] == "tpl"].drop(columns=["type"])

    sp = sp[["id", "user_id", "started_at", "finished_at", "geometry", "duration", "purpose", "detected_purpose", "overseas"]]
    tpls = tpls[["id", "user_id", "started_at", "finished_at","duration", "mode", "geometry"]]

    return sp.set_index("id"), tpls.set_index("id")

sp, tpls = filter_duplicates(sp.reset_index(), tpls.reset_index())

In [ ]:
len(sp), len(tpls)

# Read quality filter file

In [ ]:
quality_path = os.path.join("..","data", "quality")
quality_file = os.path.join("mobis_filtered.csv")
if Path(quality_file).is_file():
    valid_users = pd.read_csv(quality_file)["user_id"].values
else:
    if not os.path.exists(quality_path):
        os.makedirs(quality_path)

In [ ]:
len(valid_users)

In [ ]:
valid_users_stat = pd.read_csv(quality_file)
valid_users_stat["quality"].median() * 100, valid_users_stat["quality"].mean() * 100

In [ ]:
sp_for_days = sp.loc[sp["user_id"].isin(valid_users)]

In [ ]:
days = sp_for_days.groupby("user_id").apply(lambda x: x["finished_at"].max() - x["started_at"].min())

In [ ]:
days.dt.days.describe()

# Define activity

In [ ]:
sp["is_activity"] = True

# wait is not an activity
sp.loc[sp["purpose"] == "wait", "is_activity"] = False

# shorter than 25min
sp.loc[(sp["purpose"] == "unknown") & (sp["duration"] < 25 * 60), "is_activity"] = False

In [ ]:
sp["is_activity"].value_counts()

# Generate trips

In [ ]:
# the trackintel trip generation
sp, tpls, trips = ti.preprocessing.triplegs.generate_trips(sp, tpls, gap_threshold=25, add_geometry=False)

In [ ]:
len(sp), len(tpls), len(trips)

# Generate user filter

In [ ]:
def _split_overlaps(source, granularity="day", max_iter=60):
    if granularity == "hour":
        # every split over hour splits also over day
        # this way to split of an entry over a month takes 30+24 iterations instead of 30*24.
        df = _split_overlaps(source, granularity="day", max_iter=max_iter)
    else:
        df = source.copy()

    change_flag = _get_split_index(df, granularity=granularity)
    iter_count = 0

    freq = "D" if granularity == "day" else "H"
    # Iteratively split one day/hour from multi day/hour entries until no entry spans over multiple days/hours
    while change_flag.sum() > 0:
        # calculate new finished_at timestamp (00:00 midnight)
        new_df = df.loc[change_flag].copy()
        # print(change_flag)
        # print(new_df)
        df.loc[change_flag, "finished_at"] = (df.loc[change_flag, "started_at"] + pd.Timestamp.resolution).dt.ceil(freq)

        # create new entries with remaining timestamp
        new_df["started_at"] = df.loc[change_flag, "finished_at"]

        df = pd.concat((df, new_df), ignore_index=True, sort=True)

        change_flag = _get_split_index(df, granularity=granularity)
        iter_count += 1
        if iter_count >= max_iter:
            break

    if "duration" in df.columns:
        df["duration"] = df["finished_at"] - df["started_at"]
    return df

def _get_split_index(df, granularity="day"):
    freq = "D" if granularity == "day" else "H"
    cond1 = df["started_at"].dt.floor(freq) != (df["finished_at"] - pd.Timedelta.resolution).dt.floor(freq)
    # catch corner case where both on same border and subtracting would lead to error
    cond2 = df["started_at"] != df["finished_at"]
    return cond1 & cond2 
    
def _filter_user(df, min_thres, mean_thres):
    consider = df.loc[df["quality"] != 0]
    if (consider["quality"].min() > min_thres) and (consider["quality"].mean() > mean_thres):
        return df


def _get_tracking_quality(df, window_size):

    weeks = (df["finished_at"].max() - df["started_at"].min()).days // 7
    start_date = df["started_at"].min().date()

    quality_list = []
    # construct the sliding week gdf
    for i in range(0, weeks - window_size + 1):
        curr_start = datetime.datetime.combine(start_date + datetime.timedelta(weeks=i), datetime.time())
        curr_end = datetime.datetime.combine(curr_start + datetime.timedelta(weeks=window_size), datetime.time())

        # the total df for this time window
        cAll_gdf = df.loc[(df["started_at"] >= curr_start) & (df["finished_at"] < curr_end)]
        if cAll_gdf.shape[0] == 0:
            continue
        total_sec = (curr_end - curr_start).total_seconds()

        quality_list.append([i, cAll_gdf["duration"].sum() / total_sec])
    ret = pd.DataFrame(quality_list, columns=["timestep", "quality"])
    ret["user_id"] = df["user_id"].unique()[0]
    return ret

def calculate_user_quality(sp, trips, file_path, quality_filter):

    trips["started_at"] = pd.to_datetime(trips["started_at"]).dt.tz_localize(None)
    trips["finished_at"] = pd.to_datetime(trips["finished_at"]).dt.tz_localize(None)
    sp["started_at"] = pd.to_datetime(sp["started_at"]).dt.tz_localize(None)
    sp["finished_at"] = pd.to_datetime(sp["finished_at"]).dt.tz_localize(None)

    # merge trips and staypoints
    print("starting merge", sp.shape, trips.shape)
    sp["type"] = "sp"
    trips["type"] = "tpl"
    all_df = pd.concat([sp, trips])
    print("finished merge", all_df.shape)
    print("*" * 50)
    all_df = _split_overlaps(all_df, granularity="day")
    all_df["duration"] = (all_df["finished_at"] - all_df["started_at"]).dt.total_seconds()

    print(len(all_df["user_id"].unique()))

    # get quality
    total_quality = temporal_tracking_quality(all_df, granularity="all")
    # get tracking days
    total_quality["days"] = (
        all_df.groupby("user_id").apply(lambda x: (x["finished_at"].max() - x["started_at"].min()).days).values
    )
    # filter based on days
    user_filter_day = (
        total_quality.loc[(total_quality["days"] > quality_filter["day_filter"])]
        .reset_index(drop=True)["user_id"]
        .unique()
    )
    # filter based on sliding quality
    sliding_quality = (
        all_df.groupby("user_id")
        .apply(_get_tracking_quality, window_size=quality_filter["window_size"])
        .reset_index(drop=True)
    )

    filter_after_day = sliding_quality.loc[sliding_quality["user_id"].isin(user_filter_day)]

    if "min_thres" in quality_filter:
        # filter based on quanlity
        filter_after_day = (
            filter_after_day.groupby("user_id")
            .apply(_filter_user, min_thres=quality_filter["min_thres"], mean_thres=quality_filter["mean_thres"])
            .reset_index(drop=True)
            .dropna()
        )

    filter_after_user_quality = filter_after_day.groupby("user_id", as_index=False)["quality"].mean()

    print("final selected user", filter_after_user_quality.shape[0])
    filter_after_user_quality.to_csv(file_path, index=False)
    return filter_after_user_quality["user_id"].values

quality_filter = {"day_filter": 50, "window_size": 5, "min_thres": 0.4, "mean_thres": 0.5}
valid_users = calculate_user_quality(sp.copy().reset_index(), trips.copy().reset_index(), quality_file, quality_filter)

# Filter
## valid users

In [ ]:
sp = sp.loc[sp["user_id"].isin(valid_users)]
tpls = tpls.loc[tpls["user_id"].isin(valid_users)]
trips = trips.loc[trips["user_id"].isin(valid_users)]

In [ ]:
len(sp["user_id"].unique()), len(tpls["user_id"].unique()), len(trips["user_id"].unique())

In [ ]:
len(sp), len(tpls), len(trips)

## Switzerland records

In [ ]:
def _filter_within_swiss(stps, swissBound):
    """Spatial filtering of staypoints."""
    # save a copy of the original projection
    init_crs = stps.crs
    # project to projected system
    stps = stps.to_crs(swissBound.crs)

    ## parallel for speeding up
    stps["within"] = _apply_parallel(stps["geometry"], _apply_extract, swissBound)
    sp_swiss = stps[stps["within"] == True].copy()
    sp_swiss.drop(columns=["within"], inplace=True)

    return sp_swiss.to_crs(init_crs)
    
def _apply_extract(df, swissBound):
    """The func for _apply_parallel: judge whether inside a shp."""
    tqdm.pandas(desc="pandas bar")
    shp = swissBound["geometry"].to_numpy()[0]
    return df.progress_apply(lambda x: shp.contains(x))


def _apply_parallel(df, func, other, n=-1):
    """parallel apply for spending up."""
    if n is None:
        n = -1
    dflength = len(df)
    cpunum = multiprocessing.cpu_count()
    if dflength < cpunum:
        spnum = dflength
    if n < 0:
        spnum = cpunum + n + 1
    else:
        spnum = n or 1

    sp = list(range(dflength)[:: int(dflength / spnum + 0.5)])
    sp.append(dflength)
    slice_gen = (slice(*idx) for idx in zip(sp[:-1], sp[1:]))
    results = Parallel(n_jobs=n, verbose=0)(delayed(func)(df.iloc[slc], other) for slc in slice_gen)
    return pd.concat(results)

swissBoundary = gpd.read_file(os.path.join("..", "data", "swiss", "swiss.shp"))

print("Before spatial filtering: ", sp.shape[0])
sp_swiss = _filter_within_swiss(sp, swissBoundary)
print("After spatial filtering: ", sp_swiss.shape[0])

In [ ]:
sp_swiss = sp

## Activity staypoints

In [ ]:
sp_swiss_act = sp_swiss.loc[sp_swiss["is_activity"] == True]

In [ ]:
len(sp_swiss_act)

# Assign travel modes

In [ ]:
tpls["length"] = tpls.to_crs("EPSG:2056").length

In [ ]:
#  get the number of triplegs for each trip
groupsize = tpls.groupby("trip_id").size().to_frame(name="triplegNum").reset_index()
tpls_num = tpls.merge(groupsize, on="trip_id")

In [ ]:
# trips only with 1 triplegs
res1 = tpls_num.loc[tpls_num["triplegNum"] == 1][["trip_id", "length", "mode"]].copy()

# get the mode and length of remaining trips
remain = tpls_num.loc[tpls_num["triplegNum"] != 1].copy()

remain.sort_values(by="length", inplace=True, ascending=False)
mode = remain.groupby("trip_id").head(1).reset_index(drop=True)[["mode", "trip_id"]]

length = remain.groupby("trip_id")["length"].sum().reset_index()
res2 = mode.merge(length, on="trip_id")

# merge
res = pd.concat([res1, res2])

# cleaning
res.rename(columns={"trip_id": "id"}, inplace=True)
res.set_index("id", inplace=True)

In [ ]:
# merge to trip df
trips_mode = trips.join(res, how="left")

In [ ]:
encode_dict = {
    "Car": "Car",
    "Walk": "Walk",
    "Bicycle": "Bicycle",
    "Bus": "Bus",
    "LightRail": "Train",
    "Train": "Train",
    "Tram": "Tram",
    "RegionalTrain": "Train",
    "Ebicycle": "Bicycle",
    "MotorbikeScooter": "Car",
    "Motorbike": "Car",
    "Subway": "Tram",
    "Airplane": "Other",
    "Boat": "Other",
    "Ski": "Other",
    "TaxiUber": "Car",
    "CarsharingMobility": "Car",
    "Scooter": "Bicycle",
    "Cablecar": "Bus",
    "RidepoolingPikmi": "Car",
    "Etrottinett": "Bicycle",
    "Bikesharing": "Bicycle",
    "Escooter": "Bicycle",
    "Ferry": "Other",   
}
trips_mode["mode"] = trips_mode["mode"].apply(lambda x: encode_dict[x])

In [ ]:
trips_mode["mode"].value_counts()

In [ ]:
trips_mode.head()

In [ ]:
# combine with sp df
with_pre_trip = sp_swiss_act.loc[~sp_swiss_act["prev_trip_id"].isna()].copy()

with_pre_res = with_pre_trip.merge(trips_mode.reset_index()[["length", "mode", "id"]], how="left", left_on="prev_trip_id", right_on="id")

no_pre_trip = sp_swiss_act.loc[sp_swiss_act["prev_trip_id"].isna()].copy()
no_pre_trip["length"] = 0
no_pre_trip["mode"] = "None"

# concat result
sp_trip = pd.concat([with_pre_res, no_pre_trip]).drop(columns=["prev_trip_id", "next_trip_id", "trip_id", "id"])

In [ ]:
len(sp_swiss_act), len(sp_trip), sp_trip["mode"].value_counts()

In [ ]:
sp_trip.sort_values(by=["user_id", "started_at"], inplace=True)
sp_trip.reset_index(drop=True, inplace=True)
sp_trip.index.name = "id"

In [ ]:
sp_trip.head()

## Assign unknown travel modes

In [ ]:
def assign_unknown_modes(df):
    df.loc[df["mode"]=="None", "mode"] = pd.NA
    df["mode"] = df["mode"].ffill(axis=0).bfill(axis=0)
    return df

sp_trip_fill_mode = sp_trip.groupby("user_id", as_index=False).apply(assign_unknown_modes).reset_index(drop=True)
sp_trip_fill_mode.index.name = "id"

In [ ]:
len(sp_swiss_act), len(sp_trip_fill_mode), sp_trip_fill_mode["mode"].value_counts()

# Generate locations

In [ ]:
sp_locs, locs = sp_trip_fill_mode.as_staypoints.generate_locations(
    epsilon=20, num_samples=1, distance_metric="haversine", agg_level="dataset", n_jobs=-1
)

## Filter noise staypoints

In [ ]:
sp_filter = sp_locs.loc[~sp_locs["location_id"].isna()].copy()
print("After filter non-location staypoints: ", sp_filter.shape[0])

## Save locations

In [ ]:
locs = locs[~locs.index.duplicated(keep="first")]
filtered_locs = locs.loc[locs.index.isin(sp_filter["location_id"].unique())]

# locations without duplication, user_id have no meaning
filtered_locs.as_locations.to_csv(os.path.join("loc.csv"))
print("Location size: ", sp_filter["location_id"].unique().shape[0], filtered_locs.shape[0])

## merge staypoints

In [ ]:
sp_filter = sp_filter[["user_id", "started_at", "finished_at", "geometry", "length", "mode", "location_id"]].reset_index(drop=True)
sp_filter.index.name = "id"

sp_merged = sp_filter.as_staypoints.merge_staypoints(
    triplegs=pd.DataFrame([]), max_time_gap="1min", agg={"location_id": "first", "mode":"first", "length":"sum", "geometry": "first"}
)
print("After staypoints merging: ", sp_merged.shape[0])

In [ ]:
sp_merged.head()

# Calculate staypoint duration and activity duration

In [ ]:
sp_merged.sort_values(by=["user_id", "started_at"], inplace=True)

sp_merged["duration"] = ((sp_merged["finished_at"] - sp_merged["started_at"]).dt.total_seconds() / 60).round()

In [ ]:
def get_act_duration(df):
    df["act_duration"] = pd.NA
    df["act_duration"] = ((df["finished_at"].shift(-1) - df["finished_at"]).dt.total_seconds().shift(1) / 60).round()
    
    df["act_duration"].iloc[0] = df["duration"].iloc[0]

    return df["act_duration"]

sp_merged["act_duration"] = sp_merged.groupby("user_id").apply(get_act_duration).values

In [ ]:
sp_merged.head()

# Validate and save

In [ ]:
print("User size: ", len(sp_merged["user_id"].unique()))

In [ ]:
sp_merged.to_csv(os.path.join("sp_all.csv"))

In [ ]:
read_sp = pd.read_csv(os.path.join("sp_all.csv"), parse_dates=["started_at", "finished_at"])
print(read_sp.head())

# unique users
print(read_sp["user_id"].nunique())

In [10]:
data_activities     = Path("/data/baliu/thesis/00data/01_survey/mobis_activities.csv")
read_a = pd.read_csv(data_activities)

data_legs     = Path("/data/baliu/thesis/00data/01_survey/mobis_legs.csv")
read_legs = pd.read_csv(data_legs)

data_p = Path("/data/baliu/thesis/00data/01_survey/mobis_tracked_participants.csv")
read_p = pd.read_csv(data_p)

print(read_legs.head())

# unique users
print(read_a["user_id"].nunique())

  user_id   trip_id  next_activity_id treatment  phase            started_at  \
0   AAALY  16076422        16554731.0   Pricing      1  2019-11-23T22:59:06Z   
1   AAALY  16554729        16554756.0   Pricing      1  2019-11-24T12:05:12Z   
2   AAALY  16554730        16554756.0   Pricing      1  2019-12-02T08:56:47Z   
3   AAALY  16554755        16579822.0   Pricing      1  2019-12-02T13:41:13Z   
4   AAALY  16579818        16579823.0   Pricing      1  2019-12-02T17:58:06Z   

            finished_at  length    duration   type  ... was_confirmed  \
0  2019-11-23T23:10:55Z    4136  708.976000  Track  ...         False   
1  2019-11-24T12:05:17Z     150    5.014999  Track  ...         False   
2  2019-12-02T09:09:49Z   10533  781.656001  Track  ...         False   
3  2019-12-02T13:41:39Z     253   25.860999  Track  ...         False   
4  2019-12-02T18:14:17Z   13079  970.920001  Track  ...         False   

   in_switzerland  labeled_purpose imputed_purpose       start_x  \
0           